In [1]:
import cv2
from pathlib import Path
from ultralytics import YOLO
import shutil

# Set Path
RAW_DATA_DIR = Path("image_split")
CROPPED_DATA_DIR = Path("image_cropped")
MASKED_DATA_DIR = Path("image_masked")
FAILED_DATA_DIR = Path("image_failed")

# Load tuned yolo model
yolo_model = YOLO("yolo26_tuned_glaucoma_cropper.pt")

def process_image(task, image_path, save_path, results):
    # Check if detection failed
    if len(results[0].boxes) == 0:
        return False
        
    # Get bounding box of highest confidence detection
    box = results[0].boxes[0].xyxy[0].cpu().numpy()
    x1, y1, x2, y2 = map(int, box)
    
    # Load image
    img = cv2.imread(str(image_path))
    
    # Add padding
    padding = 20
    y1 = max(0, y1 - padding)
    y2 = min(img.shape[0], y2 + padding)
    x1 = max(0, x1 - padding)
    x2 = min(img.shape[1], x2 + padding)
    
    if task == "crop":
        cropped_img = img[y1:y2, x1:x2]
        cv2.imwrite(str(save_path), cropped_img)
    elif task == "mask":
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 0, 0), -1)
        cv2.imwrite(str(save_path), img)
    
    return True

# Recursively process all images
for img_path in RAW_DATA_DIR.rglob("*.jpg"):
    # Get relative path from RAW_DATA_DIR
    relative_path = img_path.relative_to(RAW_DATA_DIR)
    
    # Create corresponding paths in output directories
    crop_path = CROPPED_DATA_DIR / relative_path
    mask_path = MASKED_DATA_DIR / relative_path
    fail_path = FAILED_DATA_DIR / relative_path
    
    # Create parent directories if they don't exist
    crop_path.parent.mkdir(parents=True, exist_ok=True)
    mask_path.parent.mkdir(parents=True, exist_ok=True)
    fail_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Run inference ONCE per image
    results = yolo_model(str(img_path), verbose=False)
    
    # Check if detection failed
    if len(results[0].boxes) == 0:
        shutil.copy(str(img_path), str(fail_path))
        print(f"❌ Copied failed image: {relative_path}")
    else:
        # Process successful detections
        process_image("crop", img_path, crop_path, results)
        process_image("mask", img_path, mask_path, results)
        print(f"✓ Processed: {relative_path}")


✓ Processed: test\GON+\101_0.jpg
✓ Processed: test\GON+\111_0.jpg
✓ Processed: test\GON+\113_0.jpg
✓ Processed: test\GON+\116_0.jpg
✓ Processed: test\GON+\116_1.jpg
✓ Processed: test\GON+\117_0.jpg
✓ Processed: test\GON+\132_0.jpg
✓ Processed: test\GON+\132_1.jpg
✓ Processed: test\GON+\140_0.jpg
✓ Processed: test\GON+\140_1.jpg
✓ Processed: test\GON+\140_2.jpg
✓ Processed: test\GON+\143_0.jpg
✓ Processed: test\GON+\143_1.jpg
✓ Processed: test\GON+\143_2.jpg
✓ Processed: test\GON+\143_3.jpg
✓ Processed: test\GON+\143_4.jpg
✓ Processed: test\GON+\144_0.jpg
✓ Processed: test\GON+\144_1.jpg
✓ Processed: test\GON+\144_2.jpg
✓ Processed: test\GON+\144_3.jpg
✓ Processed: test\GON+\147_0.jpg
✓ Processed: test\GON+\147_1.jpg
✓ Processed: test\GON+\154_0.jpg
✓ Processed: test\GON+\154_1.jpg
✓ Processed: test\GON+\154_2.jpg
✓ Processed: test\GON+\154_3.jpg
✓ Processed: test\GON+\154_4.jpg
✓ Processed: test\GON+\154_5.jpg
✓ Processed: test\GON+\155_0.jpg
✓ Processed: test\GON+\155_1.jpg
✓ Processe